# ==================================================
# Student Productivity Analytics Project
# SQL Analytics Notebook
# ==================================================

# Goal:
# Perform SQL-based behavioral and performance analysis
# using analyst-style business queries.

In [2]:
# ==================================================
# Import Libraries
# ==================================================

import pandas as pd
import sqlite3
import os

In [3]:
# ==================================================
# Connect Google Drive
# ==================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ==================================================
# Project Paths
# ==================================================

PROJECT_PATH = "/content/drive/MyDrive/student-productivity-analytics"

PROCESSED_DATA_PATH = f"{PROJECT_PATH}/data/processed"

SQL_PATH = f"{PROJECT_PATH}/sql"

os.makedirs(SQL_PATH, exist_ok=True)

In [5]:
# ==================================================
# Load Processed Dataset
# ==================================================

dataset_path = (
    f"{PROCESSED_DATA_PATH}/"
    "student_productivity_processed.csv"
)

df = pd.read_csv(dataset_path)

print("Dataset Loaded!")

Dataset Loaded!


In [6]:
# ==================================================
# Create SQLite Database
# ==================================================

connection = sqlite3.connect("student_productivity.db")

print("Database Created!")

Database Created!


In [7]:
# ==================================================
# Export Data to SQL Table
# ==================================================

df.to_sql(
    "student_behavior",
    connection,
    if_exists="replace",
    index=False
)

print("Table Created Successfully!")

Table Created Successfully!


In [8]:
# ==================================================
# Preview SQL Table
# ==================================================

query = """
SELECT *
FROM student_behavior
LIMIT 5
"""

pd.read_sql(query, connection)

,Student_ID,Date,Persona,Age,Gender,Department,Year_of_Study,Sleep_Hours,Study_Hours,Screen_Time_Hours,...,Study_Efficiency,Wellness_Score,Productivity_Efficiency,Digital_Distraction_Score,Wellness_Index,Study_Consistency_Score,Sleep_Deficit,Burnout_Risk_Score,Performance_Category,High_Risk_Flag
0,1,2025-01-01,High Performer,18,Female,Information Technology,2,7.9,5.4,4.6,...,10.156250,3.85,8.06,2.02,3.85,8.91,0.1,3.37,Good,0
1,1,2025-01-02,High Performer,18,Female,Information Technology,2,7.9,5.0,3.5,...,8.316667,3.79,7.25,2.16,3.79,8.91,0.1,2.78,Average,0
2,1,2025-01-03,High Performer,18,Female,Information Technology,2,6.8,4.1,5.5,...,10.235294,3.56,9.82,2.83,3.56,8.91,1.2,3.32,Average,0
3,1,2025-01-04,High Performer,18,Female,Information Technology,2,7.8,4.9,3.7,...,10.847458,4.50,10.73,0.79,4.50,8.91,0.2,2.36,Good,0
4,1,2025-01-05,High Performer,18,Female,Information Technology,2,7.7,3.5,2.7,...,12.177778,4.28,11.89,1.09,4.28,8.91,0.3,2.49,Average,0


In [9]:
# ==================================================
# Average Exam Score
# ==================================================

query = """
SELECT
    AVG(Exam_Score) AS average_exam_score
FROM student_behavior
"""

pd.read_sql(query, connection)

,average_exam_score
0,39.300773


In [10]:
# ==================================================
# Student Count by Persona
# ==================================================

query = """
SELECT
    Persona,
    COUNT(*) AS total_records
FROM student_behavior
GROUP BY Persona
ORDER BY total_records DESC
"""

pd.read_sql(query, connection)

,Persona,total_records
0,Burnout,3150
1,Distracted,3090
2,Inconsistent,3060
3,Average,3000
4,High Performer,2700


In [11]:
# ==================================================
# High Burnout Students
# ==================================================

query = """
SELECT
    Student_ID,
    Burnout_Risk_Score,
    Exam_Score
FROM student_behavior
WHERE Burnout_Risk_Score > 7
ORDER BY Burnout_Risk_Score DESC
LIMIT 10
"""

pd.read_sql(query, connection)

,Student_ID,Burnout_Risk_Score,Exam_Score
0,14,8.73,8.7
1,154,8.66,0.0
2,464,8.58,50.2
3,147,8.44,25.7
4,466,8.42,24.5
5,31,8.40,7.7
6,410,8.18,48.9
7,147,8.13,45.7
8,123,8.12,25.9
9,293,8.10,0.0


In [12]:
# ==================================================
# Average Metrics by Persona
# ==================================================

query = """
SELECT
    Persona,

    ROUND(AVG(Study_Hours), 2)
        AS avg_study_hours,

    ROUND(AVG(Sleep_Hours), 2)
        AS avg_sleep_hours,

    ROUND(AVG(Exam_Score), 2)
        AS avg_exam_score,

    ROUND(AVG(Productivity_Score), 2)
        AS avg_productivity

FROM student_behavior

GROUP BY Persona

ORDER BY avg_exam_score DESC
"""

pd.read_sql(query, connection)

,Persona,avg_study_hours,avg_sleep_hours,avg_exam_score,avg_productivity
0,High Performer,5.50,7.48,62.51,56.45
1,Burnout,7.93,4.53,42.10,34.22
2,Average,4.01,6.50,41.88,37.99
3,Inconsistent,4.19,5.51,34.46,31.09
4,Distracted,2.55,6.01,18.46,19.64


In [13]:
# ==================================================
# Performance Category Analysis
# ==================================================

query = """
SELECT
    Performance_Category,
    COUNT(*) AS total_students,

    ROUND(AVG(Exam_Score), 2)
        AS avg_exam_score

FROM student_behavior

GROUP BY Performance_Category

ORDER BY avg_exam_score DESC
"""

pd.read_sql(query, connection)

,Performance_Category,total_students,avg_exam_score
0,Excellent,75,84.43
1,Good,2036,67.01
2,Average,5398,48.87
3,Poor,7491,24.42


In [14]:
# ==================================================
# Sleep Classification
# ==================================================

query = """
SELECT

    CASE

        WHEN Sleep_Hours < 5
            THEN 'Sleep Deprived'

        WHEN Sleep_Hours BETWEEN 5 AND 7
            THEN 'Moderate Sleep'

        ELSE 'Healthy Sleep'

    END AS sleep_category,

    COUNT(*) AS total_students,

    ROUND(AVG(Exam_Score), 2)
        AS avg_exam_score

FROM student_behavior

GROUP BY sleep_category
"""

pd.read_sql(query, connection)

,sleep_category,total_students,avg_exam_score
0,Healthy Sleep,3863,50.04
1,Moderate Sleep,7202,36.06
2,Sleep Deprived,3935,34.69


In [15]:
# ==================================================
# Ranking Students by Exam Score
# ==================================================

query = """
SELECT

    Student_ID,

    Persona,

    Exam_Score,

    RANK() OVER (
        ORDER BY Exam_Score DESC
    ) AS student_rank

FROM student_behavior

LIMIT 20
"""

pd.read_sql(query, connection)

,Student_ID,Persona,Exam_Score,student_rank
0,265,Inconsistent,100.0,1
1,395,Inconsistent,96.9,2
2,341,Inconsistent,95.5,3
3,138,High Performer,94.7,4
4,23,Inconsistent,94.2,5
5,466,Inconsistent,91.7,6
6,9,Inconsistent,91.4,7
7,10,Inconsistent,89.8,8
8,480,High Performer,88.9,9
9,429,High Performer,87.8,10


In [16]:
# ==================================================
# Common Table Expression (CTE)
# ==================================================

query = """
WITH persona_performance AS (

    SELECT

        Persona,

        ROUND(AVG(Exam_Score), 2)
            AS avg_exam_score

    FROM student_behavior

    GROUP BY Persona
)

SELECT *

FROM persona_performance

ORDER BY avg_exam_score DESC
"""

pd.read_sql(query, connection)

,Persona,avg_exam_score
0,High Performer,62.51
1,Burnout,42.10
2,Average,41.88
3,Inconsistent,34.46
4,Distracted,18.46


In [17]:
# ==================================================
# Top Productive Students
# ==================================================

query = """
SELECT

    Student_ID,

    Productivity_Score,

    Exam_Score,

    Burnout_Risk_Score

FROM student_behavior

ORDER BY Productivity_Score DESC

LIMIT 15
"""

pd.read_sql(query, connection)

,Student_ID,Productivity_Score,Exam_Score,Burnout_Risk_Score
0,265,75.0,100.0,2.91
1,212,74.9,86.9,2.26
2,78,72.4,76.2,1.75
3,395,72.3,96.9,4.77
4,74,72.2,86.3,1.83
5,47,71.9,75.2,1.64
6,247,71.8,74.6,2.02
7,407,71.8,79.1,2.03
8,65,71.2,82.4,2.31
9,214,71.2,86.5,5.75


In [18]:
# ==================================================
# Burnout Analysis
# ==================================================

query = """
SELECT

    Burnout_Risk,

    COUNT(*) AS total_students,

    ROUND(AVG(Exam_Score), 2)
        AS avg_exam_score,

    ROUND(AVG(Sleep_Hours), 2)
        AS avg_sleep

FROM student_behavior

GROUP BY Burnout_Risk
"""

pd.read_sql(query, connection)

,Burnout_Risk,total_students,avg_exam_score,avg_sleep
0,High,2090,38.22,4.00
1,Low,8245,44.19,6.53
2,Medium,4665,31.14,5.82


In [19]:
# ==================================================
# Save SQL Queries
# ==================================================

queries = """

-- Average Exam Score
SELECT AVG(Exam_Score)
FROM student_behavior;

-- Burnout Analysis
SELECT
    Burnout_Risk,
    AVG(Exam_Score)
FROM student_behavior
GROUP BY Burnout_Risk;

"""

query_file = (
    f"{SQL_PATH}/analysis_queries.sql"
)

with open(query_file, "w") as file:
    file.write(queries)

print("SQL File Saved!")

SQL File Saved!
